In [ ]:
# Alliance Protocol: local Python data workflow
#
# Open this notebook in VS Code and select GIS_NEA/.venv/bin/python as the kernel.
# All explanations are Python comments. Run the cells in order.
# The inputs are audited eligible transactions and saved geocoder results.
# This workflow does not infer closures, poverty, or visitor journeys.
# 1. Use the local Python environment
#
# DuckDB queries tables and geographic relationships. GeoPandas reads the tract shapefile
# and groups tracts into NTAs.
# The local .venv is already configured for this project.

# If dependencies are missing, run this in the VS Code terminal:
# .venv/bin/python -m pip install -r processing/requirements.txt
# No package installation runs automatically when this notebook opens.

In [ ]:
# 2. Find the project and load local file paths
#
# VS Code may start the notebook in the project folder or processing folder.
# The code below handles either location. Input paths are stored in
# processing/config.local.json, which is excluded from Git.
# Edit that configuration if you move the source data.

from pathlib import Path
import sys

PROJECT = Path.cwd().resolve()
if PROJECT.name == "processing":
    PROJECT = PROJECT.parent

assert (PROJECT / "processing/pipeline.py").is_file(), "Open the GIS_NEA folder in VS Code first."
sys.path.insert(0, str(PROJECT / "processing"))

from pipeline import load_config, run_pipeline
config = load_config(PROJECT / "processing/config.local.json")
config

In [ ]:
# 3. Run the pipeline
#
# The function reads source files, builds SQL tables, runs checks, and writes exports.
# Funding stays in integer cents while it is summed. Negative adjustments remain signed.
# Full polygons are used for spatial joins; display simplification happens afterward. A
# successful run also builds a database for the following queries.

manifest = run_pipeline(config, project_root=PROJECT)
manifest["funding"]

In [ ]:
# 4. Open the saved database read-only
#
# This is our query tool. Each cell below sends SQL to the same database. Read-only mode
# keeps exploration from changing the prepared tables.

import duckdb

database = PROJECT / "processing/build/analysis.duckdb"
con = duckdb.connect(str(database), read_only=True,
    config={"extension_directory": str(database.parent / "extensions")})
con.execute("LOAD spatial")

In [ ]:
# 5. Do all funding dollars reconcile?
#
# Located plus unlocated funding must equal the annual total. Unlocated dollars are not
# deleted because they cannot appear as map points.

con.sql("""
    SELECT fiscal_year, total_cents / 100.0 AS total_usd,
           located_cents / 100.0 AS located_usd,
           unlocated_cents / 100.0 AS unlocated_usd,
           total_cents = located_cents + unlocated_cents AS reconciles
    FROM accounting
    ORDER BY fiscal_year
""").df()

In [ ]:
# 6. Does the new spatial join agree with the earlier NTA assignment?
#
# A spatial join asks which polygon covers each point. We keep the previous assignment in
# the funding series and compare it to this independent check. Disagreements and boundary
# cases should be investigated, not silently rewritten.

con.sql("""
    SELECT result, COUNT(*) AS recipients
    FROM nta_assignment_audit
    GROUP BY result
""").df()

In [ ]:
con.sql("""
    SELECT id, name, recorded_nta, spatial_candidates, candidate_count
    FROM nta_assignment_audit
    WHERE result <> 'agrees' OR candidate_count > 1
    ORDER BY name
""").df()

In [ ]:
# 7. Which neighborhoods show the largest dollar losses?
#
# GROUP BY has already summed recipients into NTAs. ORDER BY sorts the result. CASE
# reports a percentage only when FY2024 funding is positive; zero or negative baselines
# are left blank. Dollars and percentages are different measures; neither establishes why
# funding changed.

con.sql("""
    SELECT name, recipient_count,
           cents2024 / 100.0 AS fy2024_usd,
           cents2025 / 100.0 AS fy2025_usd,
           (cents2025-cents2024) / 100.0 AS change_usd,
           CASE WHEN cents2024 > 0 THEN 100.0*(cents2025-cents2024)/cents2024 END AS change_percent
    FROM nta_funding
    WHERE cents2024 <> 0 OR cents2025 <> 0
    ORDER BY change_usd, name
    LIMIT 15
""").df()

In [ ]:
# 8. How many directory records are within half a mile?
#
# EPSG:2263 uses US survey feet, so half a mile is 2,640 feet. ST_DWithin performs a
# geographic distance test, rather than comparing screen pixels. These twelve anchors were
# selected by dollar decline, one per rounded coordinate. Directory duplicates and co-
# located records remain included.

con.sql("""
    SELECT a.name, a.change_cents / 100.0 AS change_usd, n.nearby_records
    FROM anchors a
    JOIN nearby_counts n USING(id)
    ORDER BY a.change_cents, a.name
""").df()

In [ ]:
# This directly repeats the spatial count so you can inspect the logic.
con.sql("""
    SELECT a.name, COUNT(d.record_id) AS nearby_records
    FROM anchors a
    LEFT JOIN directory d ON ST_DWithin(a.geom, d.geom, 2640)
    GROUP BY a.id, a.name
    ORDER BY nearby_records, a.name
""").df()

In [ ]:
# 9. What does the directory tell us about disciplines?
#
# These are DCLA directory categories. They have not been linked to the NEA funding
# records, so this query does not measure discipline-specific funding loss.

con.sql("""
    SELECT COALESCE(discipline, 'Unknown') AS discipline, COUNT(*) AS records
    FROM directory
    GROUP BY discipline
    ORDER BY records DESC
""").df()

In [ ]:
# 10. Inspect and save a query result
#
# The database is reusable: changing a question does not require geocoding or rebuilding
# every map layer. Parquet summaries and CSV audits are also saved in
# processing/build/reports/.

result = con.sql("SELECT id, name, result FROM nta_assignment_audit ORDER BY name").df()
result.to_csv(PROJECT / "processing/build/my_nta_check.csv", index=False)
con.close()

In [ ]:
# 11. Use the exports
#
# The site reads data/map-sources.json, funding.json, chapters.json, and GeoJSON files.
# JavaScript only draws and animates them. Edit the writing in content/chapters.json, then
# rerun the pipeline. Do not edit generated exports as your master copy.
#
# The full processing notes and limitations are in processing/README.md. The database and
# raw local paths are ignored by Git; public web exports can be committed through GitHub
# Desktop.
